In [1]:
import pandas as pd
import fastparquet

In [27]:
# Arquivos de Populacao

pop10 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-0a10anos-RIPSA.xlsx')
pop11a59 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-11a59-RIPSA.xlsx')
pop60 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-60+RIPSA.xlsx')
pop_geral = pd.read_excel('Dados-apoio/proj-2015-2019-POP-GERAL-RIPSA.xlsx')

pops = [pop10,pop11a59,pop60,pop_geral]

In [28]:
pop_merge = pop10.copy() # Cria uma cópia para não mexer no original

for i in pops:
    # O merge traz as colunas novas e você salva o resultado em pop_merge
    pop_merge = pop_merge.merge(
        right=i.iloc[:, [0, 2]], 
        how='left', 
        on='IBGE'
    )

# Exclui coluna desnecessaria
del pop_merge['POP10_y']

# Renomeia uma coluna
pop_merge.rename(columns={'POP10_x': 'POP10'}, inplace=True)

In [30]:
base01 = pd.read_excel('Dados-iniciais/base01.xlsx')
muni = pd.read_excel('Dados-iniciais/_Muni_por_Macro_DRS_CIR.xlsx')
sinan = pd.read_parquet('/home/usuario/Documentos/Lucas/Projetos estudos/Escorpiao_pesa/Dados-processados/2_df_recodificado.parquet', 
                        filters=[('ANO', 'in', list(range(2015, 2020)))] # Com filtro interno
)

In [42]:
df = (
    base01
    .drop_duplicates()
    .merge(
        pop_merge.drop_duplicates(),
        left_on="MUNI_REFERENCIADO",
        right_on="MUNI_NOME",
        how="left",
        indicator=True
    )
)

df

,ACESSO_LOCAL,MULTIPLO_PESA,REGIAO,PESA,MUNI_REFERENCIADO,OBSERVACOES,LAT_MUNI,LON_MUNI,LAT_PESA,LON_PESA,DISTANCIA,TEMPO,IBGE,MUNI_NOME,POP10,POP11A59,POP60,POP_GERAL,_merge
0,0,0,ARACATUBA,PENAPOLIS,ALTO ALEGRE,TODOS,-21.582059,-50.166198,-21.416404,-50.064911,23.5,25.7,350110,ALTO ALEGRE,491.0,2654.0,934.8,4079.8,both
1,0,0,ARACATUBA,PENAPOLIS,AVANHANDAVA,TODOS,-21.460333,-49.946516,-21.416404,-50.064911,14.1,15.9,350440,AVANHANDAVA,1761.2,8523.0,1383.0,11667.2,both
2,0,0,ARACATUBA,PENAPOLIS,BARBOSA,TODOS,-21.265661,-49.951816,-21.416404,-50.064911,26.7,22.8,350510,BARBOSA,975.4,4279.6,984.0,6239.0,both
3,0,0,ARACATUBA,VALPARAISO,BENTO DE ABREU,TODOS,-21.271572,-50.811723,-21.230655,-50.861195,9.5,12.8,350620,BENTO DE ABREU,426.4,1872.6,392.6,2691.6,both
4,0,0,ARACATUBA,CLEMENTINA,BILAC,TODOS,-21.403962,-50.474640,-21.556698,-50.446533,21.6,19.8,350640,BILAC,895.2,5043.4,1411.2,7349.8,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
640,0,0,SAO JOSE DO RIO PRETO,SAO JOSE DO RIO PRETO,IPIGUA,TODOS,NaN,NaN,NaN,NaN,27.3,37.0,352115,IPIGUA,882.8,3862.4,912.0,5657.2,both
641,0,0,SAO JOSE DO RIO PRETO,SAO JOSE DO RIO PRETO,ONDA VERDE,TODOS,NaN,NaN,NaN,NaN,32.4,35.0,353400,ONDA VERDE,703.4,3132.4,594.2,4430.0,both
642,1,0,PIRACICABA,PIRACICABA,PIRACICABA,TODOS,NaN,NaN,NaN,NaN,0.0,0.0,353870,PIRACICABA,58954.0,290572.8,60091.8,409618.6,both
643,0,0,PIRACICABA,PIRACICABA,RIO DAS PEDRAS,TODOS,NaN,NaN,NaN,NaN,16.4,22.0,354400,RIO DAS PEDRAS,5179.2,22896.8,3642.0,31718.0,both


In [31]:
x = sinan['NOME_MUNI'].value_counts().reset_index(name='n')
x

,NOME_MUNI,n
0,PIRACICABA,5555
1,ARACATUBA,3400
2,RIBEIRAO PRETO,2950
3,LIMEIRA,2819
4,VOTUPORANGA,2537
...,...,...
630,MONGAGUA,1
631,SANTO ANTONIO DO PINHAL,1
632,IGUAPE,1
633,BARRA DO CHAPEU,1


In [35]:
comparacao = (
    base01[["MUNI_REFERENCIADO"]]
    .drop_duplicates()
    .merge(
        pop_merge[["MUNI_NOME"]].drop_duplicates(),
        left_on="MUNI_REFERENCIADO",
        right_on="MUNI_NOME",
        how="left",
        indicator=True
    )
)

nao_encontrados = comparacao[comparacao["_merge"] == "left_only"]

print(nao_encontrados)

Empty DataFrame
Columns: [MUNI_REFERENCIADO, MUNI_NOME, _merge]
Index: []
